# Práctica 5. Mi primer RAG bajo diagnóstico

Notebook de arranque. Construye el pipeline sobre el corpus de Facturio, responde 13 preguntas del gold set, clasifica cada falla, compara unidades de chunking y decide entre prompting, RAG y fine-tuning para tres requisitos. Las celdas `TODO` traen placeholders: el notebook corre de principio a fin y las tablas salen con "(pendiente)" hasta que las llenes.

Instrucciones y rúbrica en `practice.md`. El reporte se llena en `report-template.md`.

## 0. Preparación

In [ ]:
# En Colab basta ejecutar esta celda una vez. Las semanas 5 a 7 usan las mismas librerías.
!pip install -q sentence-transformers faiss-cpu openai transformers torch

In [ ]:
# Llave de OpenAI. En Colab: panel de secretos (icono de llave) con el nombre OPENAI_API_KEY
# y acceso activado para este notebook. En Jupyter local: variable de entorno antes de abrir.
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    pass  # fuera de Colab, o sin secreto: se usa la variable de entorno si existe
print("llave cargada" if os.environ.get("OPENAI_API_KEY") else "sin llave: se usará el modelo local")

In [ ]:
import os, re, json, time, warnings, urllib.request
from pathlib import Path
import numpy as np
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# --- Corpus de Facturio: copia local del repositorio si existe; si no, se descarga de GitHub.
CORPUS_ARCHIVOS = [
    "01-manual-facturacion-cfdi.md", "02-manual-usuarios-e-invitaciones.md",
    "03-manual-reportes-y-exportacion.md", "04-manual-integraciones-y-api.md",
    "05-preguntas-frecuentes.md", "06-politica-reembolsos-y-cargos.md",
    "07-politica-cancelacion-y-eliminacion.md", "08-terminos-del-servicio-y-sla.md",
    "09-catalogo-codigos-de-error.md", "10-notas-de-version.md",
    "11-guia-de-seguridad.md", "12-comunicado-externo.md",
]
GOLD_ARCHIVO = "gold_set.jsonl"
RAW = "https://raw.githubusercontent.com/JunBarrs24/itm-maestria-llms/main/shared/datasets/corpus/"

def cargar_corpus(cache="corpus"):
    candidatos = [Path("../../shared/datasets/corpus"), Path("shared/datasets/corpus"), Path(cache)]
    base = next((b for b in candidatos if (b / CORPUS_ARCHIVOS[0]).exists()), None)
    if base is None:
        base = Path(cache); base.mkdir(exist_ok=True)
        for nombre in CORPUS_ARCHIVOS + [GOLD_ARCHIVO]:
            if not (base / nombre).exists():
                urllib.request.urlretrieve(RAW + nombre, base / nombre)
    docs = {n: (base / n).read_text(encoding="utf-8") for n in CORPUS_ARCHIVOS}
    gold = [json.loads(l) for l in (base / GOLD_ARCHIVO).read_text(encoding="utf-8").splitlines() if l.strip()]
    return docs, gold

docs, gold = cargar_corpus()
print(f"{len(docs)} documentos, {sum(len(t.split()) for t in docs.values())} palabras, {len(gold)} preguntas en el gold set")

In [ ]:
# --- Cliente mínimo. Copia de course/shared/llm.py: OpenAI Responses API si hay llave, Qwen local si no.
class LLM:
    def __init__(self, model_local="Qwen/Qwen2.5-1.5B-Instruct", model_api="gpt-5-mini"):
        self.backend = "openai" if os.environ.get("OPENAI_API_KEY") else "local"
        if self.backend == "openai":
            from openai import OpenAI
            self.client, self.model = OpenAI(), model_api
        else:
            import torch
            from transformers import AutoModelForCausalLM, AutoTokenizer
            self.tok = AutoTokenizer.from_pretrained(model_local)
            self.net = AutoModelForCausalLM.from_pretrained(model_local, dtype=torch.float32).eval()
            self.model = model_local

    def chat(self, user, system, max_output_tokens=200):
        t = time.perf_counter()
        if self.backend == "openai":
            params = dict(model=self.model, instructions=system, input=user, max_output_tokens=max_output_tokens)
            # Reasoning models (gpt-5*, o*): gastan max_output_tokens en razonar; con "minimal" la respuesta
            # cabe en presupuestos pequeños. Ejercicio: probar "low", "medium", "high" y medir tokens y latencia.
            if self.model.startswith(("gpt-5", "o1", "o3", "o4")):
                params["reasoning"] = {"effort": "minimal"}
            r = self.client.responses.create(**params)
            return {"texto": r.output_text, "in": r.usage.input_tokens, "out": r.usage.output_tokens, "s": time.perf_counter() - t}
        import torch
        msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
        texto = self.tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        ids = self.tok(texto, return_tensors="pt").input_ids
        with torch.no_grad():
            out = self.net.generate(ids, max_new_tokens=max_output_tokens, do_sample=False, pad_token_id=self.tok.eos_token_id)
        nuevos = out[0, ids.shape[1]:]
        return {"texto": self.tok.decode(nuevos, skip_special_tokens=True).strip(), "in": int(ids.shape[1]), "out": int(nuevos.shape[0]), "s": time.perf_counter() - t}

llm = LLM()
print("backend:", llm.backend, "|", llm.model)
if llm.backend == "local":
    print("Sin OPENAI_API_KEY: las respuestas de este notebook son ilustrativas (modelo local pequeño).")

## 1. Pipeline (el mismo de la sesión)

In [ ]:
def chunk_fijo(archivo, texto, tam=800, overlap=100):
    """Ventanas de `tam` caracteres con traslape. Guarda el último encabezado visto como sección."""
    chunks, seccion, pos = [], "", 0
    while pos < len(texto):
        enc = re.findall(r"^#{2,3}\s+(.*)$", texto[:pos + tam], re.M)
        if enc:
            seccion = enc[-1].strip()
        chunks.append({"archivo": archivo, "seccion": seccion, "texto": texto[pos:pos + tam]})
        pos += tam - overlap
    return chunks

def chunk_por_seccion(archivo, texto):
    """Un chunk por encabezado ## o ###: respeta la estructura que el autor ya puso."""
    chunks, seccion, buf = [], None, []
    for linea in texto.splitlines():
        m = re.match(r"^(#{2,3})\s+(.*)$", linea)
        if m:
            if buf and seccion:
                chunks.append({"archivo": archivo, "seccion": seccion, "texto": "\n".join(buf).strip()})
            seccion, buf = m.group(2).strip(), [linea]
        else:
            buf.append(linea)
    if buf and seccion:
        chunks.append({"archivo": archivo, "seccion": seccion, "texto": "\n".join(buf).strip()})
    return chunks

fijo = [c for n, t in docs.items() for c in chunk_fijo(n, t)]
seccion = [c for n, t in docs.items() for c in chunk_por_seccion(n, t)]
for nombre, cs in [("fijo 800/100", fijo), ("por sección", seccion)]:
    tam = [len(c["texto"]) for c in cs]
    print(f"{nombre:14s} {len(cs):4d} chunks   chars: media {int(np.mean(tam))}, min {min(tam)}, max {max(tam)}")
print("\nUn chunk por sección, por ejemplo:")
ej = next(c for c in seccion if c["archivo"].startswith("06") and c["seccion"] == "Plazo")
print(ej["archivo"], "#", ej["seccion"], "\n", ej["texto"][:400])

In [ ]:
from sentence_transformers import SentenceTransformer
emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")   # un encoder (semana 2)
chunks = seccion                                       # la unidad de retrieval de esta sesión
t = time.perf_counter()
E = emb.encode([c["texto"] for c in chunks], normalize_embeddings=True, batch_size=32)
print("matriz de embeddings:", E.shape, f"en {time.perf_counter()-t:.1f} s   (una fila por chunk, normalizada)")

pregunta = "¿Cuántos días tengo para solicitar el reembolso de un cargo?"
v = emb.encode([pregunta], normalize_embeddings=True)[0]
scores = E @ v                                          # producto punto = coseno, porque ambos están normalizados
orden = np.argsort(-scores)[:5]
print("\nTop-5 por coseno (índice numpy en cinco líneas):")
for i in orden:
    print(f"  {scores[i]:.3f}  {chunks[i]['archivo']} # {chunks[i]['seccion']}")

In [ ]:
import faiss
indice = faiss.IndexFlatIP(E.shape[1])      # producto interno sobre vectores normalizados = coseno
indice.add(E.astype(np.float32))
D, I = indice.search(v.astype(np.float32).reshape(1, -1), 5)
print("FAISS devuelve los mismos cinco:", [int(i) for i in I[0]] == [int(i) for i in orden])

def recuperar(pregunta, k=5):
    q = emb.encode([pregunta], normalize_embeddings=True).astype(np.float32).reshape(1, -1)
    D, I = indice.search(q, k)
    return [{**chunks[i], "score": float(d)} for d, i in zip(D[0], I[0])]

In [ ]:
SYSTEM = (
    "Eres el asistente de soporte de Facturio. Respondes únicamente con la evidencia que se te entrega, "
    "en español y en máximo tres oraciones. Al final de cada afirmación cita la fuente entre corchetes "
    "con el formato [archivo#sección]. Si la evidencia no contiene la respuesta, responde exactamente: "
    "No tengo evidencia en la documentación para responder eso."
)

def construir_contexto(chunks_rec):
    partes = []
    for i, c in enumerate(chunks_rec, start=1):
        partes.append(f'<evidencia id="{i}" fuente="{c["archivo"]}#{c["seccion"]}">\n{c["texto"]}\n</evidencia>')
    return "\n\n".join(partes)

def construir_prompt(pregunta, chunks_rec):
    return f"{construir_contexto(chunks_rec)}\n\n<pregunta>\n{pregunta}\n</pregunta>"

rec = recuperar(pregunta, k=3)
print(construir_prompt(pregunta, rec)[:1500], "\n...")

In [ ]:
log = []   # registro por llamada: tokens, latencia, fuentes

def responder(pregunta, k=5, system=SYSTEM):
    rec = recuperar(pregunta, k)
    r = llm.chat(construir_prompt(pregunta, rec), system=system)
    fuentes = [f"{c['archivo']}#{c['seccion']}" for c in rec]
    log.append({"pregunta": pregunta, "in": r["in"], "out": r["out"], "s": r["s"], "fuentes": fuentes})
    return r["texto"], fuentes

def mostrar(pregunta, k=5):
    texto, fuentes = responder(pregunta, k)
    print(f"Pregunta: {pregunta}\nRespuesta: {texto}\nFuentes top-{k}:")
    for f in fuentes:
        print("   ", f)
    print()

mostrar("¿Cuántos días tengo para solicitar el reembolso de un cargo?")

## 2. Trece preguntas del gold set

Diez con evidencia en el corpus y tres sin respuesta. La columna `evid.top-5` dice si la sección correcta llegó al modelo. Con el modelo local esto tarda unos minutos.

In [ ]:
# Preguntas de la práctica: diez con evidencia (tipos factual_exacta y conceptual) y tres sin respuesta.
IDS_CON_EVIDENCIA = ["q01", "q02", "q04", "q07", "q08", "q11", "q12", "q15", "q17", "q19"]
IDS_SIN_RESPUESTA = ["q21", "q22", "q24"]
K = 5

def evidencia_en_topk(rec, evidencia):
    return any(c["archivo"] == e["archivo"] and c["seccion"] == e["seccion"] for c in rec for e in evidencia)

resultados = []
for g in gold:
    if g["id"] not in IDS_CON_EVIDENCIA + IDS_SIN_RESPUESTA:
        continue
    rec = recuperar(g["pregunta"], K)
    texto, fuentes = responder(g["pregunta"], K)
    resultados.append({
        "id": g["id"], "tipo": g["tipo"], "pregunta": g["pregunta"],
        "evidencia_en_topk": evidencia_en_topk(rec, g["evidencia"]) if g["evidencia"] else None,
        "respuesta": texto, "esperada": g["respuesta_esperada"], "fuentes": fuentes[:3],
        "tokens_in": log[-1]["in"], "latencia_s": round(log[-1]["s"], 1),
    })

for r in resultados:
    ev = {True: "sí", False: "NO", None: "n/a"}[r["evidencia_en_topk"]]
    print(f"[{r['id']}] {r['tipo']:15s} evid.top-{K}: {ev:3s}\n   P: {r['pregunta']}\n   R: {r['respuesta'][:200]}\n   esperada: {r['esperada'][:120]}\n")

## 3. Diagnóstico por pregunta

In [ ]:
# TODO: para cada id, escribe si la respuesta fue correcta (True/False) y el tipo de falla si no lo fue:
#   "correcta", "falla_retrieval", "falla_generacion", "sin_respuesta_correcta", "sin_respuesta_inventada"
# y una línea de justificación que mencione las fuentes recuperadas.
MI_DIAGNOSTICO = {r["id"]: {"correcta": None, "falla": "", "justificacion": ""} for r in resultados}

# Tabla para el reporte
print(f"{'id':5} {'tipo':16} {'evid':5} {'correcta':9} falla")
for r in resultados:
    d = MI_DIAGNOSTICO[r["id"]]
    ev = {True: "sí", False: "NO", None: "n/a"}[r["evidencia_en_topk"]]
    print(f"{r['id']:5} {r['tipo']:16} {ev:5} {str(d['correcta']):9} {d['falla']}")

## 4. Unidad de chunking

Cuatro configuraciones sobre el mismo gold set. `hit@5 archivo` es generoso (el archivo correcto llegó); `hit@5 sección` es estricto (la sección correcta llegó). `tokens/pregunta` es lo que se paga por llamada.

In [ ]:
# Comparación de unidades de chunking con la misma métrica: ¿la sección correcta (o al menos el archivo) está en el top-5?
def construir_indice(lista):
    M = emb.encode([c["texto"] for c in lista], normalize_embeddings=True).astype(np.float32)
    ix = faiss.IndexFlatIP(M.shape[1]); ix.add(M)
    return ix, lista

configs = {"fijo 400/50": [c for n, t in docs.items() for c in chunk_fijo(n, t, 400, 50)],
           "fijo 800/100": fijo,
           "fijo 1500/200": [c for n, t in docs.items() for c in chunk_fijo(n, t, 1500, 200)],
           "por sección": seccion}
con_ev = [g for g in gold if g["evidencia"]]
print(f"{'configuración':16} {'chunks':7} {'chars':6} {'hit@5 archivo':14} {'hit@5 sección':14} {'tokens/pregunta':15}")
tabla_chunking = []
for nombre, lista in configs.items():
    ix, lista = construir_indice(lista)
    hits_a = hits_s = 0; toks = 0
    for g in con_ev:
        q = emb.encode([g["pregunta"]], normalize_embeddings=True).astype(np.float32).reshape(1, -1)
        D, I = ix.search(q, 5)
        rec = [lista[i] for i in I[0]]
        hits_a += any(c["archivo"] == e["archivo"] for c in rec for e in g["evidencia"])
        hits_s += any(c["archivo"] == e["archivo"] and c["seccion"] == e["seccion"] for c in rec for e in g["evidencia"])
        toks += sum(len(c["texto"]) for c in rec) // 4      # aproximación: 4 caracteres por token
    fila = (nombre, len(lista), int(np.mean([len(c["texto"]) for c in lista])), hits_a / len(con_ev), hits_s / len(con_ev), toks // len(con_ev))
    tabla_chunking.append(fila)
    print(f"{fila[0]:16} {fila[1]:7d} {fila[2]:6d} {fila[3]:14.2f} {fila[4]:14.2f} {fila[5]:15d}")

In [ ]:
# TODO: elige una configuración para la mesa de soporte y justifica con las dos columnas de hit y la de tokens.
MI_CONFIGURACION = ""
MI_JUSTIFICACION_CHUNKING = """
"""

## 5. Prompting, RAG o fine-tuning

In [ ]:
# TODO: para cada requisito de Facturio decide "prompting", "rag" o "fine-tuning" y justifica en una línea
# con los criterios de la sesión: ¿la información cambia?, ¿es privada?, ¿hace falta trazabilidad?, ¿es comportamiento o conocimiento?
REQUISITOS = {
    "R1": "Responder preguntas de clientes sobre políticas y límites del plan, que cambian cada versión.",
    "R2": "Redactar todas las respuestas de soporte con el tono y la estructura de la casa (saludo, diagnóstico, siguiente paso, despedida).",
    "R3": "Clasificar tickets en las cinco categorías del curso con el mínimo de latencia y costo.",
}
MI_DECISION = {
    "R1": ("", ""),   # (técnica, justificación)
    "R2": ("", ""),
    "R3": ("", ""),
}
for k, (tec, just) in MI_DECISION.items():
    print(f"{k}: {tec or '(pendiente)':12s} {REQUISITOS[k][:70]}...")

## 6. Resumen para el reporte

In [ ]:
tot_in = sum(l["in"] for l in log); tot_out = sum(l["out"] for l in log)
print(f"llamadas: {len(log)}   tokens in: {tot_in}   tokens out: {tot_out}   latencia media: {np.mean([l['s'] for l in log]):.1f} s")
print(f"tokens de contexto por pregunta (media): {tot_in/len(log):.0f}   ->  el retrieval decide cuánto pagas por llamada")

In [ ]:
print("Diagnósticos:", sum(1 for d in MI_DIAGNOSTICO.values() if d["correcta"] is not None), "de", len(MI_DIAGNOSTICO))
print("Configuración elegida:", MI_CONFIGURACION or "(pendiente)")
print("Decisiones:", {k: v[0] or "(pendiente)" for k, v in MI_DECISION.items()})